**EV Battery Failure**

In [112]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve
)

RANDOM_STATE = 42

**Load data**

In [113]:
df = pd.read_csv('/content/ev_battery_failure_dataset.csv')
print(f"Loaded {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Failure rate: {df['battery_failure'].mean():.2%}\n")

Loaded 200,000 rows, 70 columns
Failure rate: 9.96%



**Feature Selection**

In [114]:
ID_COLS = ['vehicle_id', 'battery_serial']

In [115]:
DROP_CATEGORICAL = ['vehicle_model']

In [116]:
DROP_NUMERIC_REDUNDANT = ['battery_health_percent', 'capacity_loss_percent', 'aging_score','manufacturing_year','slow_charge_ratio', 'city_driving_ratio',
    'average_ambient_temperature', 'minimum_temperature', 'maximum_temperature','thermal_health_score','battery_capacity_kwh',]

In [117]:
TARGET = 'battery_failure'

drop_cols = ID_COLS + DROP_CATEGORICAL + DROP_NUMERIC_REDUNDANT
X = df.drop(columns=drop_cols + [TARGET])
y = df[TARGET].copy()

categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols = X.select_dtypes(include='number').columns.tolist()

print(f"Kept {len(numeric_cols)} numeric features, {len(categorical_cols)} categorical features")
print(f"Dropped {len(drop_cols)} columns as IDs / redundant / low-value\n")

Kept 48 numeric features, 7 categorical features
Dropped 14 columns as IDs / redundant / low-value



**Train/Test Split**

In [118]:
X_filtered = X.copy()
y_filtered = y.copy()

# Drop rows where the target 'y' is NaN
nan_in_y_mask = y_filtered.isna()
if nan_in_y_mask.any():
    print(f"Dropping {nan_in_y_mask.sum():,} rows where target 'y' is NaN.\n")
    X_filtered = X_filtered[~nan_in_y_mask]
    y_filtered = y_filtered[~nan_in_y_mask]

X_temp, X_test, y_temp, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.15, stratify=y_filtered, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}\n")

Train: 140,080 | Val: 29,920 | Test: 30,000



**Cleaning and scalling pipeline**

In [119]:
numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),])

categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

**Model**

In [120]:
model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    learning_rate_init=1e-3,
    early_stopping=True,
    n_iter_no_change=15,
    validation_fraction=0.1,
    max_iter=300,
    random_state=RANDOM_STATE,)

pipeline = Pipeline([('prep', preprocessor),('clf', model),])

sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)

print("Training MLP classifier")
pipeline.fit(X_train, y_train)
print("Done.\n")

Training MLP classifier
Done.



**Evaluate**

In [121]:
def evaluate(name, X_eval, y_eval, threshold=0.5):
    proba = pipeline.predict_proba(X_eval)[:, 1]
    preds = (proba >= threshold).astype(int)
    print(f"--- {name} (threshold={threshold}) ---")
    print(classification_report(y_eval, preds, target_names=['healthy', 'failure'], digits=3))
    print("Confusion matrix [rows=actual, cols=predicted]:")
    print(confusion_matrix(y_eval, preds))
    roc_auc = roc_auc_score(y_eval, proba)
    pr_auc = average_precision_score(y_eval, proba)
    print(f"ROC-AUC: {roc_auc:.3f} | PR-AUC: {pr_auc:.3f}\n")
    return proba, preds, roc_auc, pr_auc

val_proba, _, _, _ = evaluate("VALIDATION @ default threshold", X_val, y_val, threshold=0.5)


--- VALIDATION @ default threshold (threshold=0.5) ---
              precision    recall  f1-score   support

     healthy      0.975     0.986     0.981     26940
     failure      0.863     0.769     0.814      2980

    accuracy                          0.965     29920
   macro avg      0.919     0.878     0.897     29920
weighted avg      0.964     0.965     0.964     29920

Confusion matrix [rows=actual, cols=predicted]:
[[26576   364]
 [  687  2293]]
ROC-AUC: 0.988 | PR-AUC: 0.912



In [122]:
precisions, recalls, thresholds = precision_recall_curve(y_val, val_proba)
target_recall = 0.90
candidates = [(t, p, r) for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds) if r >= target_recall]
if candidates:
    chosen_threshold, chosen_precision, chosen_recall = max(candidates, key=lambda x: x[0])
else:
    chosen_threshold, chosen_precision, chosen_recall = 0.5, None, None

print(f"Chosen operating threshold (targeting >= {target_recall:.0%} recall on validation): "
      f"{chosen_threshold:.3f}  (val precision {chosen_precision:.3f}, val recall {chosen_recall:.3f})\n")

Chosen operating threshold (targeting >= 90% recall on validation): 0.219  (val precision 0.736, val recall 0.900)



**Final test-set evaluation**

In [123]:
test_proba, test_preds, test_roc_auc, test_pr_auc = evaluate("TEST (held-out) default 0.5", X_test, y_test, threshold=0.5)
evaluate("TEST (held-out) chosen recall-biased threshold", X_test, y_test, threshold=chosen_threshold)

summary = {
    'n_rows': int(df.shape[0]),
    'n_features_used': len(numeric_cols) + len(categorical_cols),
    'failure_rate': float(y.mean()),
    'test_roc_auc': float(test_roc_auc),
    'test_pr_auc': float(test_pr_auc),
    'chosen_threshold': float(chosen_threshold),}


--- TEST (held-out) default 0.5 (threshold=0.5) ---
              precision    recall  f1-score   support

     healthy      0.976     0.987     0.981     27012
     failure      0.866     0.780     0.821      2988

    accuracy                          0.966     30000
   macro avg      0.921     0.883     0.901     30000
weighted avg      0.965     0.966     0.965     30000

Confusion matrix [rows=actual, cols=predicted]:
[[26650   362]
 [  657  2331]]
ROC-AUC: 0.989 | PR-AUC: 0.918

--- TEST (held-out) chosen recall-biased threshold (threshold=0.21946155406483464) ---
              precision    recall  f1-score   support

     healthy      0.989     0.965     0.977     27012
     failure      0.743     0.901     0.814      2988

    accuracy                          0.959     30000
   macro avg      0.866     0.933     0.895     30000
weighted avg      0.964     0.959     0.961     30000

Confusion matrix [rows=actual, cols=predicted]:
[[26079   933]
 [  297  2691]]
ROC-AUC: 0.989 | 

In [124]:
"""
EV Battery Failure Predictor - FastAPI service.

Loads the sklearn Pipeline (preprocessing + MLPClassifier) saved by
01_train_and_save_model.ipynb, together with a feature manifest that
records exactly which columns the pipeline expects. The web form and
the JSON API are both generated from that manifest, so this file does
not need to hardcode any of the feature names.
"""

import json
from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates

In [125]:
BASE_DIR = Path.cwd()
MODEL_DIR = BASE_DIR / "model"

In [126]:
app = FastAPI(
    title="EV Battery Failure Predictor",
    description="Predicts battery failure risk from vehicle telemetry features.",
    version="1.0.0",)

In [127]:
# Ensure directories exist
(BASE_DIR / "templates").mkdir(parents=True, exist_ok=True)
(BASE_DIR / "static").mkdir(parents=True, exist_ok=True)
(MODEL_DIR).mkdir(parents=True, exist_ok=True)

templates = Jinja2Templates(directory=str(BASE_DIR / "templates"))
app.mount("/static", StaticFiles(directory=str(BASE_DIR / "static")), name="static")

pipeline = None
manifest = None

In [128]:
@app.on_event("startup")
def load_artifacts() -> None:
    """Load the trained pipeline and feature manifest into memory."""
    global pipeline, manifest

    model_path = MODEL_DIR / "model.joblib"
    manifest_path = MODEL_DIR / "feature_manifest.json"

    if not model_path.exists() or not manifest_path.exists():
        raise RuntimeError(
            f"Could not find model artifacts in {MODEL_DIR}.\n"
            "Run 01_train_and_save_model.ipynb first, then copy the "
            "resulting model.joblib and feature_manifest.json into the "
            "model/ folder before starting the API."
        )

    pipeline = joblib.load(model_path)
    with open(manifest_path) as f:
        manifest = json.load(f)


def _coerce_row(values: dict) -> pd.DataFrame:
    """Build a single-row DataFrame matching the columns the pipeline expects."""
    row = {}

    for col in manifest["numeric_cols"]:
        raw = values.get(col)
        if raw in (None, ""):
            row[col] = None
        else:
            try:
                row[col] = float(raw)
            except (TypeError, ValueError):
                raise ValueError(f"'{col}' must be numeric, got: {raw!r}")

    for col in manifest["categorical_cols"]:
        raw = values.get(col)
        row[col] = raw if raw not in (None, "") else None

    return pd.DataFrame([row])


def _predict(values: dict):
    X = _coerce_row(values)
    proba = float(pipeline.predict_proba(X)[:, 1][0])
    threshold = manifest["chosen_threshold"]
    prediction = int(proba >= threshold)
    return proba, prediction, threshold

/tmp/ipykernel_1034/1918981086.py:1: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


In [129]:
@app.get("/", response_class=HTMLResponse)
def form_page(request: Request):
    """Simple HTML form built dynamically from the feature manifest."""
    return templates.TemplateResponse(
        "index.html",
        {
            "request": request,
            "numeric_cols": manifest["numeric_cols"],
            "categorical_cols": manifest["categorical_cols"],
            "categorical_options": manifest.get("categorical_options", {}),},)

In [135]:
"""
EV Battery Failure Predictor - FastAPI service.

Loads the sklearn Pipeline (preprocessing + MLPClassifier) saved by
01_train_and_save_model.ipynb, together with a feature manifest that
records exactly which columns the pipeline expects. The web form and
the JSON API are both generated from that manifest, so this file does
not need to hardcode any of the feature names.
"""

import json
from pathlib import Path
from contextlib import asynccontextmanager

import joblib
import pandas as pd
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates

BASE_DIR = Path.cwd()
MODEL_DIR = BASE_DIR / "model"

@asynccontextmanager
async def lifespan(app: FastAPI):
    """Load the trained pipeline and feature manifest into memory and provide them to the app state."""
    model_path = MODEL_DIR / "model.joblib"
    manifest_path = MODEL_DIR / "feature_manifest.json"

    if not model_path.exists() or not manifest_path.exists():
        raise RuntimeError(
            f"Could not find model artifacts in {MODEL_DIR}.\n"
            "Run 01_train_and_save_model.ipynb first, then copy the "
            "resulting model.joblib and feature_manifest.json into the "
            "model/ folder before starting the API."
        )

    app.state.pipeline = joblib.load(model_path)
    with open(manifest_path) as f:
        app.state.manifest = json.load(f)
    yield
    # Clean up / shut down operations (if any) go here

app = FastAPI(
    title="EV Battery Failure Predictor",
    description="Predicts battery failure risk from vehicle telemetry features.",
    version="1.0.0",
    lifespan=lifespan
)

templates = Jinja2Templates(directory=str(BASE_DIR / "templates"))
app.mount("/static", StaticFiles(directory=str(BASE_DIR / "static")), name="static")

def _coerce_row(values: dict) -> pd.DataFrame:
    """Build a single-row DataFrame matching the columns the pipeline expects."""
    row = {}

    for col in app.state.manifest["numeric_cols"]:
        raw = values.get(col)
        if raw in (None, ""):
            row[col] = None
        else:
            try:
                row[col] = float(raw)
            except (TypeError, ValueError):
                raise ValueError(f"'{col}' must be numeric, got: {raw!r}")

    for col in app.state.manifest["categorical_cols"]:
        raw = values.get(col)
        row[col] = raw if raw not in (None, "") else None

    return pd.DataFrame([row])


def _predict(values: dict):
    X = _coerce_row(values)
    proba = float(app.state.pipeline.predict_proba(X)[:, 1][0])
    threshold = app.state.manifest["chosen_threshold"]
    prediction = int(proba >= threshold)
    return proba, prediction, threshold


@app.get("/", response_class=HTMLResponse)
def form_page(request: Request):
    """Simple HTML form built dynamically from the feature manifest."""
    return templates.TemplateResponse(
        "index.html",
        {
            "request": request,
            "numeric_cols": app.state.manifest["numeric_cols"],
            "categorical_cols": app.state.manifest["categorical_cols"],
            "categorical_options": app.state.manifest.get("categorical_options", {}),
        },
    )


@app.post("/predict", response_class=HTMLResponse)
async def predict_form(request: Request):
    """Handle the HTML form submission and render a result page."""
    form = await request.form()
    values = dict(form)

    try:
        proba, prediction, threshold = _predict(values)
    except ValueError as exc:
        return templates.TemplateResponse(
            "index.html",
            {
                "request": request,
                "numeric_cols": app.state.manifest["numeric_cols"],
                "categorical_cols": app.state.manifest["categorical_cols"],
                "categorical_options": app.state.manifest.get("categorical_options", {}),
                "error": str(exc),
                "submitted": values,
            },
            status_code=400,
        )

    return templates.TemplateResponse(
        "result.html",
        {
            "request": request,
            "proba_pct": round(proba * 100, 2),
            "label": "FAILURE RISK" if prediction else "HEALTHY",
            "is_failure": bool(prediction),
            "threshold": round(threshold, 3),
        },
    )


@app.post("/api/predict")
async def predict_api(payload: dict):
    """
    JSON API endpoint.

    Send a JSON object whose keys are feature names (see GET /api/features
    for the exact list) and whose values are the corresponding readings.
    Missing keys are treated as missing values and imputed like in training.
    """
    proba, prediction, threshold = _predict(payload)
    return {
        "failure_probability": round(proba, 6),
        "prediction": "failure" if prediction else "healthy",
        "threshold_used": threshold,
    }


@app.get("/api/features")
def list_features():
    """Returns the exact feature names/types the model expects."""
    return {
        "numeric_cols": app.state.manifest["numeric_cols"],
        "categorical_cols": app.state.manifest["categorical_cols"],
        "categorical_options": app.state.manifest.get("categorical_options", {}),
    }


@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": app.state.pipeline is not None}

In [137]:
from pathlib import Path

BASE_DIR = Path.cwd()
templates_dir = BASE_DIR / "templates"
templates_dir.mkdir(parents=True, exist_ok=True)

index_html_content = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>EV Battery Failure Predictor</title>
  <link rel="stylesheet" href="/static/style.css">
</head>
<body>
  <div class="wrap">
    <h1>EV Battery Failure Predictor</h1>
    <p class="subtitle">Enter a reading below to estimate battery failure risk.</p>

    <div class="card">
      {% if error %}
        <div class="error">{{ error }}</div>
      {% endif %}

      <form action="/predict" method="post">
        {% if numeric_cols %}
        <div class="section-title">Numeric features</div>
        <div class="grid">
          {% for col in numeric_cols %}
          <div class="field">
            <label for="{{ col }}">{{ col }}</label>
            <input type="number" step="any" id="{{ col }}" name="{{ col }}"
                   value="{{ submitted.get(col, '') if submitted else '' }}">
          </div>
          {% endfor %}
        </div>
        {% endif %}

        {% if categorical_cols %}
        <div class="section-title">Categorical features</div>
        <div class="grid">
          {% for col in categorical_cols %}
          <div class="field">
            <label for="{{ col }}">{{ col }}</label>
            {% if categorical_options.get(col) %}
            <select id="{{ col }}" name="{{ col }}">
              <option value="">-- select --</option>
              {% for opt in categorical_options.get(col, []) %}
              <option value="{{ opt }}" {% if submitted and submitted.get(col) == opt %}selected{% endif %}>{{ opt }}</option>
              {% endfor %}
            </select>
            {% else %}
            <input type="text" id="{{ col }}" name="{{ col }}"
                   value="{{ submitted.get(col, '') if submitted else '' }}">
            {% endif %}
          </div>
          {% endfor %}
        </div>
        {% endif %}

        <button type="submit">Predict failure risk</button>
      </form>
    </div>
  </div>
</body>
</html>"""

with open(templates_dir / "index.html", "w") as f:
    f.write(index_html_content)

print("index.html saved to templates directory.")

index.html saved to templates directory.


In [139]:
from pathlib import Path

BASE_DIR = Path.cwd()
templates_dir = BASE_DIR / "templates"
templates_dir.mkdir(parents=True, exist_ok=True)

result_html_content = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Prediction Result</title>
  <link rel="stylesheet" href="/static/style.css">
</head>
<body>
  <div class="wrap">
    <h1>EV Battery Failure Predictor</h1>
    <div class="card result">
      <div class="badge {{ 'failure' if is_failure else 'healthy' }}">
        {{ label }}
      </div>
      <div class="proba">{{ proba_pct }}%</div>
      <div class="meta">estimated failure probability</div>
      <div class="meta" style="margin-top:10px;">
        Decision threshold in use: {{ threshold }}
      </div>
      <a class="back" href="/">&larr; Run another prediction</a>
    </div>
  </div>
</body>
</html>"""

with open(templates_dir / "result.html", "w") as f:
    f.write(result_html_content)

print("result.html saved to templates directory.")

result.html saved to templates directory.


In [141]:
from pathlib import Path

BASE_DIR = Path.cwd()
static_dir = BASE_DIR / "static"
static_dir.mkdir(parents=True, exist_ok=True)

style_css_content = """:root {
  --bg: #0f172a;
  --card: #ffffff;
  --accent: #2563eb;
  --danger: #dc2626;
  --ok: #16a34a;
  --text: #1e293b;
  --muted: #64748b;
}

* { box-sizing: border-box; }

body {
  margin: 0;
  font-family: -apple-system, \"Segoe UI\", Roboto, Helvetica, Arial, sans-serif;
  background: var(--bg);
  color: var(--text);
  padding: 32px 16px 64px;
}

.wrap { max-width: 780px; margin: 0 auto; }
h1 { color: #f8fafc; font-size: 1.6rem; margin-bottom: 4px; }
.subtitle { color: #94a3b8; margin-bottom: 24px; font-size: 0.95rem; }

.card {
  background: var(--card);
  border-radius: 12px;
  padding: 24px;
  box-shadow: 0 10px 30px rgba(0,0,0,0.25);
}

.section-title {
  font-weight: 600;
  margin: 20px 0 10px;
  color: var(--muted);
  text-transform: uppercase;
  font-size: 0.75rem;
  letter-spacing: 0.05em;
}

.grid {
  display: grid;
  grid-template-columns: repeat(auto-fill, minmax(220px, 1fr));
  gap: 12px;
}

.field label { display: block; font-size: 0.8rem; color: var(--muted); margin-bottom: 4px; }
.field input, .field select {
  width: 100%;
  padding: 8px 10px;
  border: 1px solid #cbd5e1;
  border-radius: 8px;
  font-size: 0.9rem;
}

button {
  margin-top: 24px;
  background: var(--accent);
  color: white;
  border: none;
  padding: 12px 20px;
  border-radius: 8px;
  font-size: 1rem;
  cursor: pointer;
}
button:hover { opacity: 0.9; }

.error {
  background: #fee2e2;
  color: var(--danger);
  padding: 10px 14px;
  border-radius: 8px;
  margin-bottom: 16px;
  font-size: 0.9rem;
}

.result { text-align: center; padding: 40px 20px; }
.result .badge {
  display: inline-block;
  padding: 10px 22px;
  border-radius: 999px;
  font-weight: 700;
  font-size: 1.1rem;
  margin-bottom: 16px;
}
.badge.failure { background: #fee2e2; color: var(--danger); }
.badge.healthy { background: #dcfce7; color: var(--ok); }

.proba { font-size: 2.4rem; font-weight: 700; margin: 8px 0; }
.meta { color: var(--muted); font-size: 0.85rem; }
a.back { display: inline-block; margin-top: 24px; color: var(--accent); text-decoration: none; }"""

with open(static_dir / "style.css", "w") as f:
    f.write(style_css_content)

print("style.css saved to static directory.")

style.css saved to static directory.


In [1]:
!pip install fastapi==0.115.0
!pip install uvicorn[standard]==0.30.6
!pip install scikit-learn==1.5.2
!pip install pandas==2.2.2
!pip install numpy==1.26.4
!pip install joblib==1.4.2
!pip install jinja2==3.1.4
!pip install python-multipart==0.0.9

  Using cached scikit_learn-1.5.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
Using cached scikit_learn-1.5.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.9 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.44 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. T

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: joblib
    Found existing installation: joblib 1.5.3
    Uninstalling joblib-1.5.3:
      Successfully uninstalled joblib-1.5.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.44 requires scikit-learn>=1.6, which is not installed.
pysal 25.7 requires scikit-learn>=1.1, which is not installed.
pynndescent 0.6.0 requires scikit-learn>=0.18, which is not installed.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, which is not installed.
umap-learn 0.5.12 requires scikit-learn>=1.6, which is not installed.
segregation 2.5.4 requires scikit-learn>=0.21.3, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
imbalanced-learn 0.14.2 requires scikit-learn<2,>=1.4.2, which is not installed.
tobler 0.14.0 requires numpy>=

In [3]:
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /code

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app ./app
COPY model ./model

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

SyntaxError: invalid syntax (3636548935.py, line 1)

In [4]:
dockerfile_content = """FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /code

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app ./app
COPY model ./model

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

print("Dockerfile created successfully.")

Dockerfile created successfully.


In [14]:
###: Create `requirements.txt`

In [7]:
requirements_content = """fastapi==0.115.0
uvicorn[standard]==0.30.6
scikit-learn==1.5.2
pandas==2.2.2
numpy==1.26.4
joblib==1.4.2
jinja2==3.1.4
python-multipart==0.0.9"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

print("requirements.txt created successfully.")

requirements.txt created successfully.


In [15]:
# Save pipeline and manifest to the 'model' directory
import joblib
import json
from pathlib import Path

MODEL_DIR_LOCAL = Path("model") # This refers to a local directory if you run this outside Colab, or creates 'model' in Colab's current working directory.
MODEL_DIR_LOCAL.mkdir(parents=True, exist_ok=True)

try:
    joblib.dump(pipeline, MODEL_DIR_LOCAL / "model.joblib")
except NameError:
    print("Error: The 'pipeline' variable is not defined. Please ensure you have run the model training cell (e.g., cell 5256pRIvJY-G) before attempting to save the model.")
    print("Model artifacts were NOT saved.")
else:
    # Only proceed to save manifest if pipeline was successfully dumped
    with open(MODEL_DIR_LOCAL / "feature_manifest.json", "w") as f:
        # Use the global 'summary' dictionary as the manifest
        # These variables (numeric_cols, categorical_cols, summary) are also expected from previous cells
        try:
            manifest_to_save = {
                "numeric_cols": numeric_cols,
                "categorical_cols": categorical_cols,
                "chosen_threshold": summary["chosen_threshold"],
                # Add other relevant summary items to manifest if needed
                "failure_rate": summary["failure_rate"],
                "n_features_used": summary["n_features_used"],
                "n_rows": summary["n_rows"],
                "test_pr_auc": summary["test_pr_auc"],
                "test_roc_auc": summary["test_roc_auc"]
            }
            json.dump(manifest_to_save, f)
            print("Model artifacts saved.")
        except NameError as e:
            print(f"Error saving manifest: {e}. Ensure all required variables (numeric_cols, categorical_cols, summary) are defined from previous cells.")


Error: The 'pipeline' variable is not defined. Please ensure you have run the model training cell (e.g., cell 5256pRIvJY-G) before attempting to save the model.
Model artifacts were NOT saved.


In [16]:
### Prepare `templates` and `static` directories

In [17]:
### Build the Docker Image

In [18]:
### Run the Docker Container

In [20]:
compose_content = """version: "3.9"

services:
  battery-api:
    build: .
    image: ev-battery-failure-api:latest
    container_name: ev-battery-failure-api
    ports:
      - "8000:8000"
    volumes:
      - ./model:/code/model
    restart: unless-stopped"""

with open("docker-compose.yml", "w") as f:
    f.write(compose_content)

print("docker-compose.yml created successfully.")

docker-compose.yml created successfully.
